# 4) Inside PyPSA: the optimisation model, custom constraints and Pyomo

## Objectives

At the end of this notebook you should be able to:

- open the **optimisation model PyPSA builds** (`n.model`) and map every variable and constraint to the model you wrote yourself in lesson 3
- read prices and other duals directly from PyPSA's model
- add a **custom constraint** that PyPSA does not offer — here a take-or-pay gas contract — and value it with its dual
- write the same LP in **Pyomo**, the other widely used modelling language, and compare the two tools on correctness, code style and speed

Lesson 3 showed that your own Linopy model reproduces PyPSA exactly. This lesson turns that around: you look *inside* PyPSA, and because you know what to expect, nothing in it is a black box. That is what lets you extend PyPSA when a study needs something it does not have.

In [1]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import pyomo.environ as pyo
import pypsa

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("pyomo").setLevel(logging.WARNING)      # Pyomo logs the full HiGHS output at INFO level
warnings.filterwarnings("ignore", category=FutureWarning)

### Lesson 3's day as a PyPSA network

The same sunny day: demand, free solar and run-of-river, three thermal plants with ramp limits and a cyclic 300 MW / 1,200 MWh battery.

In [2]:
T = 24
hours = np.arange(T)
load = 1900 + 400 * np.sin(np.pi * (hours - 6) / 12).clip(0) + 250 * np.exp(-((hours - 19) / 2.0) ** 2)
free = 2200 * 0.8 * np.clip(np.sin(np.pi * (hours - 6) / 13), 0, None) + 400.0
plants = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "ramp": [60, 300, 500]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))
BATTERY = {"p_nom": 300, "e_nom": 1200, "eta": 0.95}


def network(load=load, free=free, snapshots=hours):
    """Lesson 3's system as a PyPSA network (any number of snapshots)."""
    n = pypsa.Network()
    n.set_snapshots(snapshots)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, row in plants.iterrows():
        n.add("Generator", gen, bus="HR", p_nom=row.p_nom, marginal_cost=row.mc,
              ramp_limit_up=row.ramp / row.p_nom, ramp_limit_down=row.ramp / row.p_nom)
    n.add("StorageUnit", "battery", bus="HR", p_nom=BATTERY["p_nom"], max_hours=BATTERY["e_nom"] / BATTERY["p_nom"],
          efficiency_store=BATTERY["eta"], efficiency_dispatch=BATTERY["eta"], cyclic_state_of_charge=True)
    return n

### Opening the box: `n.optimize.create_model()`

`n.optimize()` does two things: it **builds** a Linopy model from the network's components, then **solves** it. You can do the two steps separately and look at the model in between:

In [3]:
n = network()
m = n.optimize.create_model()
print(f"{m.nvars} variables, {m.ncons} constraints\n")
sizes = pd.DataFrame({"kind": ["variable"] * len(m.variables) + ["constraint"] * len(m.constraints),
                      "name": list(m.variables) + list(m.constraints),
                      "entries": [m.variables[v].size for v in m.variables] + [m.constraints[c].size for c in m.constraints]})
sizes

168 variables, 522 constraints



          kind                                   name  entries
0     variable                            Generator-p       96
1     variable                 StorageUnit-p_dispatch       24
2     variable                    StorageUnit-p_store       24
3     variable            StorageUnit-state_of_charge       24
4   constraint                  Generator-fix-p-lower       96
5   constraint                  Generator-fix-p-upper       96
6   constraint              Generator-p-ramp_limit_up       96
7   constraint            Generator-p-ramp_limit_down       96
8   constraint       StorageUnit-fix-p_dispatch-lower       24
9   constraint       StorageUnit-fix-p_dispatch-upper       24
10  constraint          StorageUnit-fix-p_store-lower       24
11  constraint          StorageUnit-fix-p_store-upper       24
12  constraint  StorageUnit-fix-state_of_charge-lower       24
13  constraint  StorageUnit-fix-state_of_charge-upper       24
14  constraint                      Bus-nodal_balance  

Every name maps onto something you wrote yourself in lesson 3:

| PyPSA name | Lesson 3 |
|---|---|
| `Generator-p` (snapshot × name) | `g` for the three plants **and** `free` — PyPSA treats the free energy as a fourth generator |
| `StorageUnit-p_store`, `-p_dispatch`, `-state_of_charge` | `charge`, `discharge`, `soc` |
| `Bus-nodal_balance` | `balance` — its dual is the price |
| `Generator-p-ramp_limit_up` / `_down` | `ramp_up` / `ramp_down` |
| `StorageUnit-energy_balance` | `soc_balance` |
| `…-fix-…-lower` / `-upper` | the variable **bounds** (`lower=`, `upper=`) |

The variable count is identical: **168**, as in lesson 3 (4 generators + 3 battery variables, × 24 hours). The constraint count is not — 522 here — only because PyPSA writes each variable's bounds as explicit constraints where lesson 3 used Linopy's `lower`/`upper`. Same LP, different bookkeeping.

One detail that trips people up: inside `n.model`, the component dimension is called **`name`**, not `Generator`. Look at one ramp constraint:

In [4]:
print(m.variables["Generator-p"].dims)
m.constraints["Generator-p-ramp_limit_up"].sel(name="coal", snapshot=16)

('snapshot', 'name')


Constraint `Generator-p-ramp_limit_up`
--------------------------------------
+1 Generator-p[16, coal] - 1 Generator-p[15, coal] ≤ 60.0

It reads exactly like lesson 3's ramp constraint for coal at 16:00: this hour's output minus the previous hour's output, at most 60 MW.

### Solve, and read duals straight from the model

In [5]:
n.optimize.solve_model(solver_name="highs", log_to_console=False, progress=False)
nodal_dual = n.model.constraints["Bus-nodal_balance"].dual.sel(name="HR").to_numpy()
assert np.allclose(nodal_dual, n.buses_t.marginal_price["HR"].to_numpy()), "the price is the dual of the nodal balance"
assert np.isclose(n.objective, 1_837_742.65, atol=0.01), "lesson 3's cost"
print(f"cost {n.objective:,.2f} EUR (lesson 3: 1,837,742.65 EUR); n.buses_t.marginal_price = dual of Bus-nodal_balance")

cost 1,837,742.65 EUR (lesson 3: 1,837,742.65 EUR); n.buses_t.marginal_price = dual of Bus-nodal_balance


`n.buses_t.marginal_price` is nothing more than the dual of `Bus-nodal_balance`, copied into a table. Every other dual — of the ramp limits, the battery's energy balance — is in `n.model` too, ready for the KKT analysis of lesson 3.

### Adding a constraint PyPSA does not have

Suppose the CCGT's owner has a **take-or-pay** gas contract: the gas is paid for whether it is used or not, so the plant must burn at least enough to produce **12,000 MWh** over the day. PyPSA has no attribute for that, but because the model is ordinary Linopy, you can add the constraint yourself. `extra_functionality` is a function PyPSA calls after building the model and before solving it:

In [6]:
def take_or_pay(minimum_MWh):
    """Return an extra_functionality that forces the CCGT to produce at least minimum_MWh over the horizon."""
    def add(n, snapshots):
        p = n.model.variables["Generator-p"]
        n.model.add_constraints(p.sel(name="ccgt").sum() >= minimum_MWh, name="ccgt_take_or_pay")
    return add


rows = {}
for minimum in [8_000, 11_999, 12_000, 12_001]:
    nc = network()
    nc.optimize(solver_name="highs", log_to_console=False, progress=False, extra_functionality=take_or_pay(minimum))
    rows[minimum] = {"cost_EUR": nc.objective, "ccgt_MWh": nc.generators_t.p["ccgt"].sum(),
                     "contract_dual": float(nc.model.constraints["ccgt_take_or_pay"].dual.item()) + 0.0,
                     "night_price_03h": nc.buses_t.marginal_price["HR"].iloc[3]}
contract = pd.DataFrame.from_dict(rows, orient="index").rename_axis("minimum_MWh")
finite_difference = (contract.loc[12_001, "cost_EUR"] - contract.loc[11_999, "cost_EUR"]) / 2
assert np.isclose(finite_difference, contract.loc[12_000, "contract_dual"])
print(f"extra cost of the last contracted MWh: {finite_difference:.2f} EUR (dual: {contract.loc[12_000, 'contract_dual']:.2f})")
contract.round(2)

extra cost of the last contracted MWh: 35.00 EUR (dual: 35.00)


               cost_EUR  ccgt_MWh  contract_dual  night_price_03h
minimum_MWh                                                      
8000         1837742.65   8662.58            0.0             95.0
11999        1948103.86  11999.00           35.0             60.0
12000        1948138.86  12000.00           35.0             60.0
12001        1948173.86  12001.00           35.0             60.0

Without the contract the CCGT produces 8,663 MWh, so a minimum of 8,000 MWh changes nothing (dual 0, complementary slackness). A minimum of 12,000 MWh binds and raises the day's cost by 110,396 €. Its dual, **35 €/MWh**, is the cost of the last contracted MWh — confirmed by finite differences — and it has a plain meaning: each forced MWh of gas at 95 €/MWh displaces coal at 60 €/MWh.

Now look at the night price: it **falls** from 95 to 60 €/MWh. With gas forced on regardless, coal becomes the marginal plant at night. Forcing an expensive plant to run *lowers* the market price while *raising* the total cost — the same mechanism by which must-run obligations and take-or-pay contracts depress wholesale prices in real markets.

### The same LP in Pyomo

**Pyomo** is the other modelling language you will meet in energy research — many published models (and teaching material) use it. It writes models as **indexed sets** and **rules**: a constraint is a Python function called once per index. Duals are read back through a `Suffix`:

In [7]:
def pyomo_dispatch(load, free):
    """Lesson 3's LP (ramps + cyclic battery) in Pyomo."""
    n_t = len(load)
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, n_t - 1)
    m.G = pyo.Set(initialize=list(plants.index))
    m.g = pyo.Var(m.G, m.T, bounds=lambda m, g, t: (0, plants.loc[g, "p_nom"]))
    m.f = pyo.Var(m.T, bounds=lambda m, t: (0, free[t]))
    m.c = pyo.Var(m.T, bounds=(0, BATTERY["p_nom"]))
    m.d = pyo.Var(m.T, bounds=(0, BATTERY["p_nom"]))
    m.e = pyo.Var(m.T, bounds=(0, BATTERY["e_nom"]))
    m.balance = pyo.Constraint(m.T, rule=lambda m, t: sum(m.g[g, t] for g in m.G) + m.f[t] + m.d[t] - m.c[t] == load[t])
    m.soc = pyo.Constraint(m.T, rule=lambda m, t:
                           m.e[t] == m.e[(t - 1) % n_t] + BATTERY["eta"] * m.c[t] - m.d[t] / BATTERY["eta"])
    m.ramp_up = pyo.Constraint(m.G, m.T, rule=lambda m, g, t:
                               pyo.Constraint.Skip if t == 0 else m.g[g, t] - m.g[g, t - 1] <= plants.loc[g, "ramp"])
    m.ramp_down = pyo.Constraint(m.G, m.T, rule=lambda m, g, t:
                                 pyo.Constraint.Skip if t == 0 else m.g[g, t - 1] - m.g[g, t] <= plants.loc[g, "ramp"])
    m.cost = pyo.Objective(expr=sum(plants.loc[g, "mc"] * m.g[g, t] for g in m.G for t in m.T))
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m


pm = pyomo_dispatch(load, free)
pyo.SolverFactory("appsi_highs").solve(pm)
pyomo_prices = np.array([pm.dual[pm.balance[t]] for t in range(T)])
gap = np.abs(pyomo_prices - n.buses_t.marginal_price["HR"].to_numpy()).max()
assert np.isclose(pyo.value(pm.cost), n.objective, rtol=1e-9) and gap < 1e-6
print(f"Pyomo cost {pyo.value(pm.cost):,.2f} EUR = PyPSA {n.objective:,.2f} EUR; largest price difference {gap:.1e} EUR/MWh")

Pyomo cost 1,837,742.65 EUR = PyPSA 1,837,742.65 EUR; largest price difference 0.0e+00 EUR/MWh


Three tools — your Linopy model in lesson 3, PyPSA, and now Pyomo — give the same cost and the same 24 prices. Compare the styles: Linopy writes the ramp constraint once for all hours and plants as an array expression (`g - g.shift(t=1) <= r`); Pyomo writes a rule for a single (plant, hour) and calls it 72 times. The Pyomo form reads like the mathematics on paper; the Linopy form is closer to NumPy.

### Does the difference matter? A whole year

Repeat the day 365 times and build and solve both versions. Your timings will differ from anyone else's — what matters is the comparison on the same machine:

In [8]:
load_year, free_year = np.tile(load, 365), np.tile(free, 365)
t0 = time.perf_counter()
pm_year = pyomo_dispatch(load_year, free_year)
t1 = time.perf_counter()
pyo.SolverFactory("appsi_highs").solve(pm_year)
t2 = time.perf_counter()
n_year = network(load_year, free_year, pd.RangeIndex(len(load_year)))
n_year.optimize.create_model()
t3 = time.perf_counter()
n_year.optimize.solve_model(solver_name="highs", log_to_console=False, progress=False)
t4 = time.perf_counter()

assert np.isclose(pyo.value(pm_year.cost), n_year.objective, rtol=1e-9)
assert np.isclose(n_year.objective, 365 * n.objective, rtol=1e-9)
pd.DataFrame({"build_s": [t1 - t0, t3 - t2], "solve_s": [t2 - t1, t4 - t3]},
             index=["Pyomo", "PyPSA / Linopy"]).round(2)

                build_s  solve_s
Pyomo              1.39     3.35
PyPSA / Linopy     0.57     1.70

Both find the same optimum on 8,760 hours — exactly 365 times the daily cost, because the optimal day starts and ends in the same state (coal at full output, the battery cyclic), so repeating it is optimal. The table shows where the time goes. Linopy creates each constraint family as one array operation, while Pyomo calls a Python rule for every single index — here about 70,000 constraints, one Python call each. That per-index overhead typically makes Pyomo's model building the slower part as models grow; for a national model with many years, regions and technologies, the difference can grow from seconds to minutes, which is one reason PyPSA moved to Linopy.

### Which tool when?

| | Linopy | Pyomo | PyPSA |
|---|---|---|---|
| Style | array expressions (like NumPy/xarray) | indexed sets and rules (like the maths) | components (generators, lines, storage) |
| Strength | speed and memory on large time series | flexibility, nonlinear and MILP extensions, a large ecosystem | energy-system data model, network physics, built-in constraints |
| Use it for | your own models, extending PyPSA | reading and reproducing published models | system studies — the course's backbone |

You now know all three well enough to move between them — and to check one against another.

## Check your understanding

**PyPSA's model**

1. Why does PyPSA's model have more constraints than lesson 3's, but exactly the same number of variables? Would the two models give different duals for the bounds?
2. The take-or-pay contract has a dual of 35 €/MWh. What would it be if the CCGT displaced the OCGT instead of coal in the extra hours?

**Tools**

1. Write, in words, how you would add a CO₂ budget for the day to PyPSA using `extra_functionality` — which variable, which coefficients, which sense?
2. A colleague sends you a published Pyomo model. How would you check that your PyPSA version of the same system is equivalent?